# 🔬 Experimento: Chunking Semántico con LangChain

En este notebook exploramos el `SemanticChunker` de LangChain, que divide texto en chunks basándose en la **similitud semántica** entre oraciones, en lugar de simplemente cortar por cantidad de caracteres o tokens.

## ¿Cómo funciona?

1. Divide el texto en oraciones.
2. Genera embeddings para cada oración.
3. Calcula la similitud coseno entre oraciones consecutivas.
4. Cuando la similitud cae por debajo de un umbral (hay un cambio de tema), crea un nuevo chunk.

## Estrategias de breakpoint disponibles

| Estrategia | Descripción |
|---|---|
| `percentile` | Corta donde la diferencia de similitud supera el percentil X |
| `standard_deviation` | Corta donde la diferencia supera la media + X desviaciones estándar |
| `interquartile` | Corta usando el rango intercuartílico |
| `gradient` | Usa el gradiente de cambio en similitud |

## 1. Instalación de dependencias

In [ ]:
# Instalar dependencias necesarias
%pip install langchain-experimental langchain-openai langchain sentence-transformers --quiet

## 2. Imports

In [1]:
import os
import textwrap
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

print("✅ Imports OK")

✅ Imports OK


## 3. Configurar el modelo de embeddings

Usamos `sentence-transformers/all-MiniLM-L6-v2` (gratuito, corre localmente, sin necesidad de API key).

In [2]:
# Modelo de embeddings local (sin API key)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print(f"✅ Modelo de embeddings cargado: {embedding_model.model_name}")

C:\Users\lauty\AppData\Local\Temp\ipykernel_24872\931897613.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
c:\Users\lauty\Desktop\Facultad\Quinto año\Tesina\pdf-transformation-benchmark\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\lauty\Desktop\Facultad\Quinto año\Tesina\pdf-transformation-benchmark\env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently sto

✅ Modelo de embeddings cargado: sentence-transformers/all-MiniLM-L6-v2


## 4. Texto de prueba

Usamos un texto con **3 temas claramente distintos** para ver si el chunker los separa correctamente.

In [3]:
sample_text = """
Machine learning is a subset of artificial intelligence that enables computers to learn from data.
It involves training algorithms on large datasets to recognize patterns and make predictions.
Common machine learning techniques include supervised learning, unsupervised learning, and reinforcement learning.
Neural networks are particularly powerful for tasks like image recognition and natural language processing.
Deep learning, a subset of machine learning, uses multiple layers of neural networks to extract complex features from data.

The Amazon rainforest is the world's largest tropical rainforest, covering over 5.5 million square kilometers.
It is home to an estimated 10% of all species on Earth, many of which have not yet been discovered.
The forest plays a crucial role in regulating the global climate by absorbing carbon dioxide.
Deforestation is a major threat to the Amazon, driven by agriculture, logging, and urban expansion.
Conservation efforts are underway to protect this vital ecosystem from further destruction.

Quantum computing harnesses the principles of quantum mechanics to perform computations.
Unlike classical bits, quantum bits (qubits) can exist in superposition, representing both 0 and 1 simultaneously.
This allows quantum computers to solve certain problems exponentially faster than classical computers.
Applications include cryptography, drug discovery, and optimization problems.
Companies like IBM, Google, and startups are racing to build fault-tolerant quantum computers.
"""

print(f"📄 Texto cargado: {len(sample_text)} caracteres, {len(sample_text.split('.'))} oraciones aprox.")

📄 Texto cargado: 1520 caracteres, 17 oraciones aprox.


## 5. Función auxiliar para visualizar chunks

In [4]:
def display_chunks(chunks, strategy_name):
    """Muestra los chunks de forma legible."""
    print(f"\n{'='*70}")
    print(f"  Estrategia: {strategy_name}")
    print(f"  Total de chunks generados: {len(chunks)}")
    print(f"{'='*70}")
    
    for i, chunk in enumerate(chunks):
        print(f"\n--- Chunk {i+1} ({len(chunk.page_content)} chars) ---")
        # Wrap texto para mejor lectura
        wrapped = textwrap.fill(chunk.page_content.strip(), width=70)
        print(wrapped)
    
    print(f"\n{'='*70}\n")

## 6. Estrategia 1: Percentile (default)

Corta donde la diferencia de similitud entre oraciones consecutivas supera el percentil indicado.

In [5]:
chunker_percentile = SemanticChunker(
    embeddings=embedding_model,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90  # Corta en el percentil 90 de diferencias
)

chunks_percentile = chunker_percentile.create_documents([sample_text])
display_chunks(chunks_percentile, "Percentile (threshold=90)")


  Estrategia: Percentile (threshold=90)
  Total de chunks generados: 3

--- Chunk 1 (416 chars) ---
Machine learning is a subset of artificial intelligence that enables
computers to learn from data. It involves training algorithms on large
datasets to recognize patterns and make predictions. Common machine
learning techniques include supervised learning, unsupervised
learning, and reinforcement learning. Neural networks are particularly
powerful for tasks like image recognition and natural language
processing.

--- Chunk 2 (528 chars) ---
Deep learning, a subset of machine learning, uses multiple layers of
neural networks to extract complex features from data. The Amazon
rainforest is the world's largest tropical rainforest, covering over
5.5 million square kilometers. It is home to an estimated 10% of all
species on Earth, many of which have not yet been discovered. The
forest plays a crucial role in regulating the global climate by
absorbing carbon dioxide. Deforestation is a major 

## 7. Estrategia 2: Standard Deviation

Corta donde la diferencia de similitud supera: `media + threshold * desviación_estándar`.

In [6]:
chunker_std = SemanticChunker(
    embeddings=embedding_model,
    breakpoint_threshold_type="standard_deviation",
    breakpoint_threshold_amount=1.5  # Corta a 1.5 desviaciones estándar
)

chunks_std = chunker_std.create_documents([sample_text])
display_chunks(chunks_std, "Standard Deviation (threshold=1.5)")


  Estrategia: Standard Deviation (threshold=1.5)
  Total de chunks generados: 2

--- Chunk 1 (945 chars) ---
Machine learning is a subset of artificial intelligence that enables
computers to learn from data. It involves training algorithms on large
datasets to recognize patterns and make predictions. Common machine
learning techniques include supervised learning, unsupervised
learning, and reinforcement learning. Neural networks are particularly
powerful for tasks like image recognition and natural language
processing. Deep learning, a subset of machine learning, uses multiple
layers of neural networks to extract complex features from data. The
Amazon rainforest is the world's largest tropical rainforest, covering
over 5.5 million square kilometers. It is home to an estimated 10% of
all species on Earth, many of which have not yet been discovered. The
forest plays a crucial role in regulating the global climate by
absorbing carbon dioxide. Deforestation is a major threat to the
Amazon

## 8. Estrategia 3: Interquartile

Usa el rango intercuartílico para detectar outliers en la similitud (cambios abruptos de tema).

In [7]:
chunker_iqr = SemanticChunker(
    embeddings=embedding_model,
    breakpoint_threshold_type="interquartile",
    breakpoint_threshold_amount=1.5
)

chunks_iqr = chunker_iqr.create_documents([sample_text])
display_chunks(chunks_iqr, "Interquartile (threshold=1.5)")


  Estrategia: Interquartile (threshold=1.5)
  Total de chunks generados: 1

--- Chunk 1 (1518 chars) ---
Machine learning is a subset of artificial intelligence that enables
computers to learn from data. It involves training algorithms on large
datasets to recognize patterns and make predictions. Common machine
learning techniques include supervised learning, unsupervised
learning, and reinforcement learning. Neural networks are particularly
powerful for tasks like image recognition and natural language
processing. Deep learning, a subset of machine learning, uses multiple
layers of neural networks to extract complex features from data. The
Amazon rainforest is the world's largest tropical rainforest, covering
over 5.5 million square kilometers. It is home to an estimated 10% of
all species on Earth, many of which have not yet been discovered. The
forest plays a crucial role in regulating the global climate by
absorbing carbon dioxide. Deforestation is a major threat to the
Amazon, dr

## 9. Estrategia 4: Gradient

Analiza el gradiente del cambio de similitud para detectar puntos de quiebre.

In [8]:
chunker_gradient = SemanticChunker(
    embeddings=embedding_model,
    breakpoint_threshold_type="gradient",
    breakpoint_threshold_amount=95
)

chunks_gradient = chunker_gradient.create_documents([sample_text])
display_chunks(chunks_gradient, "Gradient (threshold=95)")


  Estrategia: Gradient (threshold=95)
  Total de chunks generados: 2

--- Chunk 1 (99 chars) ---
Machine learning is a subset of artificial intelligence that enables
computers to learn from data.

--- Chunk 2 (1418 chars) ---
It involves training algorithms on large datasets to recognize
patterns and make predictions. Common machine learning techniques
include supervised learning, unsupervised learning, and reinforcement
learning. Neural networks are particularly powerful for tasks like
image recognition and natural language processing. Deep learning, a
subset of machine learning, uses multiple layers of neural networks to
extract complex features from data. The Amazon rainforest is the
world's largest tropical rainforest, covering over 5.5 million square
kilometers. It is home to an estimated 10% of all species on Earth,
many of which have not yet been discovered. The forest plays a crucial
role in regulating the global climate by absorbing carbon dioxide.
Deforestation is a major th

## 10. Comparativa de estrategias

In [9]:
import pandas as pd

results = [
    {
        "Estrategia": "Percentile (90)",
        "N° Chunks": len(chunks_percentile),
        "Chars promedio/chunk": round(sum(len(c.page_content) for c in chunks_percentile) / len(chunks_percentile)),
        "Chunk más corto (chars)": min(len(c.page_content) for c in chunks_percentile),
        "Chunk más largo (chars)": max(len(c.page_content) for c in chunks_percentile),
    },
    {
        "Estrategia": "Std Deviation (1.5)",
        "N° Chunks": len(chunks_std),
        "Chars promedio/chunk": round(sum(len(c.page_content) for c in chunks_std) / len(chunks_std)),
        "Chunk más corto (chars)": min(len(c.page_content) for c in chunks_std),
        "Chunk más largo (chars)": max(len(c.page_content) for c in chunks_std),
    },
    {
        "Estrategia": "Interquartile (1.5)",
        "N° Chunks": len(chunks_iqr),
        "Chars promedio/chunk": round(sum(len(c.page_content) for c in chunks_iqr) / len(chunks_iqr)),
        "Chunk más corto (chars)": min(len(c.page_content) for c in chunks_iqr),
        "Chunk más largo (chars)": max(len(c.page_content) for c in chunks_iqr),
    },
    {
        "Estrategia": "Gradient (95)",
        "N° Chunks": len(chunks_gradient),
        "Chars promedio/chunk": round(sum(len(c.page_content) for c in chunks_gradient) / len(chunks_gradient)),
        "Chunk más corto (chars)": min(len(c.page_content) for c in chunks_gradient),
        "Chunk más largo (chars)": max(len(c.page_content) for c in chunks_gradient),
    },
]

df = pd.DataFrame(results)
print("📊 Comparativa de estrategias de chunking semántico:")
df

📊 Comparativa de estrategias de chunking semántico:


,Estrategia,N° Chunks,Chars promedio/chunk,Chunk más corto (chars),Chunk más largo (chars)
0,Percentile (90),3,505,416,572
1,Std Deviation (1.5),2,758,572,945
2,Interquartile (1.5),1,1518,1518,1518
3,Gradient (95),2,758,99,1418


## 11. Probá con tu propio texto o PDF

Reemplazá `my_text` con el contenido de tu PDF extraído con Docling o PyMuPDF.

In [ ]:
# ---- Probá tu propio texto ----
my_text = """
Pegá acá el texto de tu PDF...
"""

# Elegí la estrategia que mejor funcionó arriba
my_chunker = SemanticChunker(
    embeddings=embedding_model,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90
)

my_chunks = my_chunker.create_documents([my_text])
display_chunks(my_chunks, "Tu texto personalizado")

## 12. Integración con un PDF real usando PyMuPDF

In [ ]:
import fitz  # PyMuPDF
from pathlib import Path

# Cambiar a la ruta de tu PDF
PDF_PATH = Path("inputs/Informe 59 - Senado.pdf")  # Ajustá el path

# Listá los PDFs disponibles
pdf_files = list(PDF_PATH.glob("*.pdf"))
print(f"PDFs encontrados en '{PDF_PATH}':")
for p in pdf_files:
    print(f"  - {p.name}")

if not pdf_files:
    print("⚠️  No se encontraron PDFs. Colocá un PDF en la carpeta 'inputs/'")

PDFs encontrados en 'inputs\Informe 59 - Senado.pdf':
⚠️  No se encontraron PDFs. Colocá un PDF en la carpeta 'inputs/'


In [ ]:
# Si hay PDFs disponibles, procesá el primero
if pdf_files:
    pdf_path = pdf_files[0]
    print(f"Procesando: {pdf_path.name}")
    
    # Extraer texto con PyMuPDF
    doc = fitz.open(str(pdf_path))
    pdf_text = ""
    for page in doc:
        pdf_text += page.get_text()
    doc.close()
    
    print(f"📄 Texto extraído: {len(pdf_text)} caracteres")
    
    # Chunking semántico
    pdf_chunker = SemanticChunker(
        embeddings=embedding_model,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=90
    )
    
    pdf_chunks = pdf_chunker.create_documents([pdf_text])
    
    print(f"✅ Chunks generados: {len(pdf_chunks)}")
    print(f"   Promedio de chars por chunk: {round(sum(len(c.page_content) for c in pdf_chunks) / len(pdf_chunks))}")
    
    # Ver los primeros 3 chunks
    display_chunks(pdf_chunks[:3], f"Primeros 3 chunks de '{pdf_path.name}'")